# Full EQAS


In [ ]:
%pip install -q torch==2.13.0 transformers==5.16.1 datasets==5.0.1 requests \
openai chromadb langchain langchain-community langchain-openai langchain-chroma neo4j pandas


In [ ]:
import json, random, re, string, time
from pathlib import Path
from collections import Counter
import numpy as np
import torch
from datasets import Dataset, DatasetDict

SEED=42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DATA_REPO="https://github.com/Gokcimen/Home_Appliance_Dataset"
!rm -rf /content/Home_Appliance_Dataset
!git clone -q --depth 1 {DATA_REPO}.git /content/Home_Appliance_Dataset

DATA_ROOT=Path("/content/Home_Appliance_Dataset")

def flatten(path):
    raw=json.loads(Path(path).read_text(encoding="utf-8"))
    rows=[]
    for article in raw["data"]:
        title=article["title"]
        for para in article["paragraphs"]:
            context=para["context"]
            for qa in para["qas"]:
                rows.append({
                    "id":str(qa["id"]),
                    "title":title,
                    "context":context,
                    "question":qa["question"],
                    "answers":{
                        "text":[a["text"] for a in qa["answers"]],
                        "answer_start":[int(a["answer_start"]) for a in qa["answers"]],
                    }
                })
    return rows

raw_datasets=DatasetDict({
    "train":Dataset.from_list(flatten(DATA_ROOT/"train.json")),
    "validation":Dataset.from_list(flatten(DATA_ROOT/"dev.json")),
    "test":Dataset.from_list(flatten(DATA_ROOT/"test.json")),
})

assert len(raw_datasets["train"])==8000
assert len(raw_datasets["validation"])==1000
assert len(raw_datasets["test"])==1000

ids={s:set(raw_datasets[s]["id"]) for s in raw_datasets}
assert not ids["train"]&ids["validation"]
assert not ids["train"]&ids["test"]
assert not ids["validation"]&ids["test"]
all_ids=set().union(*ids.values())
assert len(all_ids)==10000
assert {int(x) for x in all_ids}==set(range(1,10001))

titles={s:set(raw_datasets[s]["title"]) for s in raw_datasets}
assert not titles["train"]&titles["validation"]
assert not titles["train"]&titles["test"]
assert not titles["validation"]&titles["test"]
assert len(set().union(*titles.values()))==1111

print("train",len(raw_datasets["train"]))
print("validation",len(raw_datasets["validation"]))
print("test",len(raw_datasets["test"]))
print("products",len(set().union(*titles.values())))


In [ ]:
THRESHOLD=0.90

def select_route(confidence,kg_available=False,retrieval_available=False):
    confidence=float(confidence)
    if confidence>=THRESHOLD:
        return "direct_qa"
    if kg_available:
        return "kg_grounded"
    if retrieval_available:
        return "retrieval_grounded"
    return "generative_fallback"

print(select_route(0.99))
print(select_route(0.23,kg_available=True))


In [ ]:
from transformers import pipeline

QA_CHECKPOINT="TunahanGokcimen/Question-Answering-Bert-base-cased-squad2"
qa_model=pipeline(
    "question-answering",
    model=QA_CHECKPOINT,
    device=0 if torch.cuda.is_available() else -1,
)

def structured_facts(title,context):
    facts=[f"Product: {title}"]
    sentences=[x.strip() for x in re.split(r"(?<=[.!?])\s+",context) if x.strip()]
    for s in sentences:
        if any(k in s.lower() for k in (
            "capacity","energy","efficiency","noise","dimension","weight",
            "feature","technology","warranty","program","kg","litre","liter","db"
        )):
            facts.append(s)
    return "\n".join(facts)


In [ ]:
import os, openai
openai.api_key=os.getenv("OPENAI_API_KEY")

def gpt_answer(question,evidence,model="gpt-4-0314"):
    response=openai.ChatCompletion.create(
        model=model,
        messages=[
            {"role":"system","content":"Use only the supplied evidence for factual product claims."},
            {"role":"user","content":f"Evidence:\n{evidence}\n\nQuestion:\n{question}"}
        ],
        temperature=0.5,
        top_p=1.0,
        n=1,
        max_tokens=512,
        stream=False,
        presence_penalty=0.1,
        frequency_penalty=0.5,
    )
    return response.choices[0].message["content"]


In [ ]:
def answer_eqas(example,retrieved_text=""):
    qa=qa_model(question=example["question"],context=example["context"])
    kg_text=structured_facts(example["title"],example["context"])

    route=select_route(
        qa["score"],
        kg_available=bool(kg_text),
        retrieval_available=bool(retrieved_text),
    )

    if route=="direct_qa":
        answer=qa["answer"]
    else:
        evidence=example["context"]
        if route=="kg_grounded":
            evidence += "\n\nStructured KG evidence:\n" + kg_text
        if route=="retrieval_grounded":
            evidence += "\n\nRetrieved evidence:\n" + retrieved_text
        answer=gpt_answer(example["question"],evidence)

    return {
        "id":example["id"],
        "answer":answer,
        "route":route,
        "qa_confidence":float(qa["score"]),
    }
